# Reconocimiento Facial  - Reconoce-Tec

In [ ]:
# 1. Montar drive

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Importar librerías

import os
import cv2
import json
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print("TensorFlow:", tf.__version__)
print("OpenCV:", cv2.__version__)

In [ ]:
# 3. Configuración de rutas

VIDEO_DIR = Path("/content/drive/MyDrive/dataset_videos")

WORK_DIR = Path("/content/drive/MyDrive/reconocimiento_rostros_work")
FRAMES_DIR = WORK_DIR / "frames"
FACES_DIR = WORK_DIR / "rostros"
DATASET_DIR = WORK_DIR / "dataset_final"

TRAIN_DIR = DATASET_DIR / "train"
VAL_DIR = DATASET_DIR / "val"

OUTPUT_DIR = Path("/content/drive/MyDrive/modelos_reconocimiento_rostros")


# No borra dataset_videos.
FORCE_CLEAN_GENERATED = True

if not VIDEO_DIR.exists():
    raise FileNotFoundError(f"No existe VIDEO_DIR: {VIDEO_DIR}")

def limpiar_carpeta(path: Path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

if FORCE_CLEAN_GENERATED:
    print("Limpiando carpetas generadas...")
    for p in [WORK_DIR, OUTPUT_DIR]:
        if p.exists():
            shutil.rmtree(p)

for p in [FRAMES_DIR, FACES_DIR, TRAIN_DIR, VAL_DIR, OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("VIDEO_DIR:", VIDEO_DIR)
print("WORK_DIR:", WORK_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
# 4. Detectar videos y clases esperadas

EXTENSIONES_VIDEO = [".mp4", ".mov", ".avi", ".mkv", ".webm"]

videos = []
for ext in EXTENSIONES_VIDEO:
    videos.extend(VIDEO_DIR.glob(f"*{ext}"))
    videos.extend(VIDEO_DIR.glob(f"*{ext.upper()}"))

videos = sorted(set(videos))

class_ids_from_videos = [v.stem for v in videos]

print("Videos encontrados:", len(videos))
print("Clases detectadas desde nombres de videos:")
for i, v in enumerate(videos):
    print(f"{i}: {v.name} -> {v.stem}")

if len(videos) < 2:
    raise ValueError("Se requieren al menos 2 videos/clases para entrenar.")

if len(set(class_ids_from_videos)) != len(class_ids_from_videos):
    raise ValueError("Hay videos con nombres duplicados. Cada archivo debe tener un ID único.")

print("\nTotal de clases esperadas:", len(class_ids_from_videos))

In [ ]:
# 5. Extraer frames desde videos

# Para videos cortos de 4 a 10 segundos:
FRAME_STEP = 5
MAX_FRAMES_PER_VIDEO = 80

limpiar_carpeta(FRAMES_DIR)

total_frames = 0
frames_por_clase = {}

for video_path in videos:
    clase = video_path.stem
    salida_clase = FRAMES_DIR / clase
    salida_clase.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        print(f"No se pudo abrir: {video_path}")
        continue

    frame_idx = 0
    guardados = 0

    while True:
        ok, frame = cap.read()

        if not ok:
            break

        if frame_idx % FRAME_STEP == 0:
            nombre_frame = f"{clase}_{frame_idx:06d}.jpg"
            cv2.imwrite(str(salida_clase / nombre_frame), frame)

            guardados += 1
            total_frames += 1

            if guardados >= MAX_FRAMES_PER_VIDEO:
                break

        frame_idx += 1

    cap.release()

    frames_por_clase[clase] = guardados
    print(f"{clase}: {guardados} frames")

print("\nTotal de frames extraídos:", total_frames)

if total_frames == 0:
    raise RuntimeError("No se extrajo ningún frame. Revisa VIDEO_DIR y los formatos de video.")

In [ ]:
# 6. Detectar y recortar rostros con OpenCV

limpiar_carpeta(FACES_DIR)

IMAGE_SIZE = 160
MARGIN = 0.25

MIN_NEIGHBORS = 3

cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
face_cascade = cv2.CascadeClassifier(cascade_path)

if face_cascade.empty():
    raise RuntimeError("No se pudo cargar Haar Cascade.")

def recortar_rostro(image_bgr, caja):
    x, y, w, h = caja
    alto_img, ancho_img, _ = image_bgr.shape

    mx = int(w * MARGIN)
    my = int(h * MARGIN)

    x1 = max(0, x - mx)
    y1 = max(0, y - my)
    x2 = min(ancho_img, x + w + mx)
    y2 = min(alto_img, y + h + my)

    if x2 <= x1 or y2 <= y1:
        return None

    crop = image_bgr[y1:y2, x1:x2]
    crop = cv2.resize(crop, (IMAGE_SIZE, IMAGE_SIZE))
    return crop

conteo_rostros = {}
total_rostros = 0
total_no_detectados = 0

for clase_dir in sorted([p for p in FRAMES_DIR.iterdir() if p.is_dir()]):
    clase = clase_dir.name
    salida_clase = FACES_DIR / clase
    salida_clase.mkdir(parents=True, exist_ok=True)

    imagenes = list(clase_dir.glob("*.jpg"))

    guardados = 0
    no_detectados = 0

    for img_path in tqdm(imagenes, desc=f"Rostros {clase}"):
        image_bgr = cv2.imread(str(img_path))
        if image_bgr is None:
            continue

        gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)

        rostros = face_cascade.detectMultiScale(
            gray,
            scaleFactor=1.1,
            minNeighbors=MIN_NEIGHBORS,
            minSize=(40, 40),
        )

        if len(rostros) == 0:
            no_detectados += 1
            total_no_detectados += 1
            continue

        # rostro más grande
        caja = max(rostros, key=lambda r: r[2] * r[3])
        rostro = recortar_rostro(image_bgr, caja)

        if rostro is None:
            no_detectados += 1
            total_no_detectados += 1
            continue

        cv2.imwrite(str(salida_clase / img_path.name), rostro)
        guardados += 1
        total_rostros += 1

    conteo_rostros[clase] = {
        "rostros_guardados": guardados,
        "frames_sin_rostro": no_detectados,
        "frames_totales": len(imagenes),
    }

print("\nRostros guardados:", total_rostros)
print("Frames sin rostro detectado:", total_no_detectados)
print(json.dumps(conteo_rostros, indent=2, ensure_ascii=False))

if total_rostros == 0:
    raise RuntimeError("No se detectó ningún rostro. Revisa videos o detector.")

In [ ]:
# 7. Ver muestras de rostros recortados

def mostrar_muestras(carpeta, n=21):
    imagenes = []
    clases = sorted([p for p in carpeta.iterdir() if p.is_dir()])

    for clase_dir in clases:
        imgs = list(clase_dir.glob("*.jpg"))
        random.shuffle(imgs)
        imagenes.extend(imgs[:max(1, n // max(1, len(clases)))])

    random.shuffle(imagenes)
    imagenes = imagenes[:n]

    if not imagenes:
        print("No hay imágenes para mostrar.")
        return

    cols = 4
    rows = int(np.ceil(len(imagenes) / cols))
    plt.figure(figsize=(12, 3 * rows))

    for i, img_path in enumerate(imagenes):
        img_bgr = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        plt.subplot(rows, cols, i + 1)
        plt.imshow(img_rgb)
        plt.title(img_path.parent.name, fontsize=8)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

mostrar_muestras(FACES_DIR, n=21)

In [ ]:
# Balancear clases

MAX_ROSTROS_POR_CLASE = 26

import random

for clase_dir in sorted([p for p in FACES_DIR.iterdir() if p.is_dir()]):
    imagenes = list(clase_dir.glob("*.jpg"))

    if len(imagenes) > MAX_ROSTROS_POR_CLASE:
        random.shuffle(imagenes)

        eliminar = imagenes[MAX_ROSTROS_POR_CLASE:]

        for img in eliminar:
            img.unlink()

        print(f"{clase_dir.name}: {len(imagenes)} -> {MAX_ROSTROS_POR_CLASE}")
    else:
        print(f"{clase_dir.name}: {len(imagenes)}")
        

In [ ]:
# 8. Validar conteos por clase antes de entrenar

min_rostros_recomendado = 20

clases_con_rostros = sorted([p.name for p in FACES_DIR.iterdir() if p.is_dir() and len(list(p.glob("*.jpg"))) > 0])

print("Clases esperadas:", class_ids_from_videos)
print("Clases con rostros:", clases_con_rostros)

faltantes = sorted(set(class_ids_from_videos) - set(clases_con_rostros))
if faltantes:
    raise RuntimeError(f"Estas clases no tienen rostros detectados: {faltantes}")

for clase in clases_con_rostros:
    cantidad = len(list((FACES_DIR / clase).glob("*.jpg")))
    print(clase, "->", cantidad, "rostros")
    if cantidad < min_rostros_recomendado:
        print(f"  ADVERTENCIA: {clase} tiene pocos rostros. Recomendado mínimo: {min_rostros_recomendado}")

print("\nTodas las clases tienen al menos un rostro.")

In [ ]:

limpiar_carpeta(DATASET_DIR)
TRAIN_DIR.mkdir(parents=True, exist_ok=True)
VAL_DIR.mkdir(parents=True, exist_ok=True)

VAL_SPLIT = 0.2
random.seed(42)

conteo_dataset = {}

for clase in class_ids_from_videos:
    clase_dir = FACES_DIR / clase
    imagenes = list(clase_dir.glob("*.jpg"))
    random.shuffle(imagenes)

    if len(imagenes) < 2:
        raise RuntimeError(f"La clase {clase} necesita al menos 2 imágenes para train/val.")

    n_val = max(1, int(len(imagenes) * VAL_SPLIT))
    n_val = min(n_val, len(imagenes) - 1)

    val_imgs = imagenes[:n_val]
    train_imgs = imagenes[n_val:]

    (TRAIN_DIR / clase).mkdir(parents=True, exist_ok=True)
    (VAL_DIR / clase).mkdir(parents=True, exist_ok=True)

    for img_path in train_imgs:
        shutil.copy(img_path, TRAIN_DIR / clase / img_path.name)

    for img_path in val_imgs:
        shutil.copy(img_path, VAL_DIR / clase / img_path.name)

    conteo_dataset[clase] = {
        "train": len(train_imgs),
        "val": len(val_imgs),
        "total": len(imagenes)
    }

print(json.dumps(conteo_dataset, indent=2, ensure_ascii=False))

# Validación estricta
for clase, datos in conteo_dataset.items():
    if datos["train"] == 0 or datos["val"] == 0:
        raise RuntimeError(f"La clase {clase} quedó sin train o sin val: {datos}")

In [ ]:
# 10. Cargar dataset con TensorFlow

BATCH_SIZE = 16
IMG_SIZE = (160, 160)
SEED = 42

train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True,
    seed=SEED,
)

val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False,
    seed=SEED,
)

class_names = train_ds_raw.class_names
num_classes = len(class_names)

print("class_names:", class_names)
print("num_classes:", num_classes)

if class_names != sorted(class_ids_from_videos):
    print("ADVERTENCIA: class_names no coincide exactamente con videos ordenados.")
    print("class_names:", class_names)
    print("videos:", sorted(class_ids_from_videos))

if num_classes != len(class_ids_from_videos):
    raise RuntimeError(f"El dataset tiene {num_classes} clases, pero se esperaban {len(class_ids_from_videos)}.")

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds_raw.prefetch(AUTOTUNE)
val_ds = val_ds_raw.prefetch(AUTOTUNE)

In [ ]:
# 11. Guardar clases.json desde class_names

clases_path = OUTPUT_DIR / "clases.json"

with open(clases_path, "w", encoding="utf-8") as f:
    json.dump(class_names, f, ensure_ascii=False, indent=2)

print("clases.json guardado en:", clases_path)
print(json.dumps(class_names, indent=2, ensure_ascii=False))

In [ ]:
# 12. Crear modelo



data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10),
], name="data_augmentation")

base_model = MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

inputs = layers.Input(shape=(160, 160, 3), name="input_image")
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.35)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.25)(x)
outputs = layers.Dense(num_classes, activation="softmax", name="salida")(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

print("model.output_shape:", model.output_shape)
if model.output_shape[-1] != num_classes:
    raise RuntimeError("La salida del modelo no coincide con num_classes.")

In [ ]:
# 13. Entrenar modelo

best_model_path = str(OUTPUT_DIR / "mejor_modelo.keras")

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=3,
        min_lr=1e-6
    ),
    ModelCheckpoint(
        best_model_path,
        monitor="val_loss",
        save_best_only=True
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks,
)

print("Épocas ejecutadas:", len(history.history["loss"]))
print("Último accuracy:", history.history["accuracy"][-1])
print("Último val_accuracy:", history.history["val_accuracy"][-1])

In [ ]:

loss, acc = model.evaluate(val_ds)
print(f"Loss validación: {loss:.4f}")
print(f"Accuracy validación: {acc:.4f}")

y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print("\nReporte de clasificación:")
print(classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    zero_division=0
))

reporte = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

reporte_path = OUTPUT_DIR / "reporte_clasificacion.json"
with open(reporte_path, "w", encoding="utf-8") as f:
    json.dump(reporte, f, ensure_ascii=False, indent=2)

print("Reporte guardado:", reporte_path)

In [ ]:
# 15. Matriz de confusión

cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, xticks_rotation=45, cmap="Blues")
plt.title("Matriz de confusión")
plt.tight_layout()

matriz_path = OUTPUT_DIR / "matriz_confusion.png"
plt.savefig(matriz_path, dpi=150)
plt.show()

print("Matriz guardada:", matriz_path)

In [ ]:

for images, labels in val_ds.take(1):
    preds = model.predict(images, verbose=0)

    plt.figure(figsize=(12, 8))

    for i in range(min(12, images.shape[0])):
        pred_idx = int(np.argmax(preds[i]))
        real_idx = int(np.argmax(labels[i]))

        plt.subplot(3, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(
            f"Real: {class_names[real_idx]}\nPred: {class_names[pred_idx]} ({preds[i][pred_idx]:.2f})",
            fontsize=8,
        )
        plt.axis("off")

    plt.tight_layout()
    plt.show()
    break

In [ ]:
# 17. Guardar modelo Keras

keras_path = OUTPUT_DIR / "modelo_clasificador.keras"
model.save(keras_path)

print("Modelo Keras guardado:", keras_path)

In [ ]:
# 18. Convertir a TensorFlow Lite

converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

tflite_path = OUTPUT_DIR / "modelo_clasificador.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

print("Modelo TFLite guardado:", tflite_path)
print("Tamaño MB:", tflite_path.stat().st_size / (1024 * 1024))

In [ ]:
# 19. Verificar TFLite: input/output y clases

interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("INPUT DETAILS:")
print(input_details)

print("\nOUTPUT DETAILS:")
print(output_details)

input_shape = input_details[0]["shape"]
output_shape = output_details[0]["shape"]

print("\nInput shape:", input_shape)
print("Output shape:", output_shape)

if output_shape[-1] != len(class_names):
    raise RuntimeError(
        f"ERROR: TFLite devuelve {output_shape[-1]} clases, pero clases.json tiene {len(class_names)}."
    )

if list(input_shape[1:]) != [160, 160, 3]:
    raise RuntimeError(f"ERROR: Input TFLite inesperado: {input_shape}")

print("\nOK: TFLite y clases.json coinciden.")

In [ ]:

def predecir_tflite_imagen_uint8(img_uint8):
    # img_uint8: [160,160,3] valores 0–255.
    input_data = np.expand_dims(img_uint8.astype(np.float32), axis=0)

    interpreter.set_tensor(input_details[0]["index"], input_data)
    interpreter.invoke()

    preds = interpreter.get_tensor(output_details[0]["index"])[0]
    return preds

correctas = 0
total = 0

for clase in class_names:
    imgs = list((VAL_DIR / clase).glob("*.jpg"))
    random.shuffle(imgs)

    print(f"\nClase real: {clase}")

    for img_path in imgs[:5]:
        img_bgr = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_rgb = cv2.resize(img_rgb, (160, 160))

        preds = predecir_tflite_imagen_uint8(img_rgb)
        idx = int(np.argmax(preds))
        pred_clase = class_names[idx]
        conf = float(preds[idx])

        total += 1
        if pred_clase == clase:
            correctas += 1

        print(f"{img_path.name} -> pred: {pred_clase}, conf: {conf:.4f}")

print("\nAccuracy muestra TFLite:", correctas / max(1, total))

In [ ]:

from google.colab import files
from PIL import Image

uploaded = files.upload()

for filename in uploaded.keys():
    img_pil = Image.open(filename).convert("RGB")
    img_pil_resized = img_pil.resize((160, 160))

    img_arr = np.array(img_pil_resized).astype(np.uint8)

    preds = predecir_tflite_imagen_uint8(img_arr)
    idx = int(np.argmax(preds))

    print("\n======================")
    print("Archivo:", filename)
    print("Predicción:", class_names[idx])
    print("Confianza:", float(preds[idx]))
    print("\nProbabilidades:")

    for c, p in zip(class_names, preds):
        print(c, "=>", round(float(p), 4))

    plt.imshow(img_pil)
    plt.axis("off")
    plt.title(f"Predicción: {class_names[idx]} | Confianza: {float(preds[idx]):.2f}")
    plt.show()